In [1]:
import cv2
import mediapipe as mp
import numpy as np
import time
import platform
import os

mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

def play_beep():
    if platform.system() == 'Windows':
        import winsound
        winsound.Beep(1000, 150)
    else:
        duration = 0.15
        freq = 1000
        os.system(f'play -nq -t alsa synth {duration} sine {freq}')

def calculate_angle(a, b, c):
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    if angle > 180.0:
        angle = 360 - angle
    return angle

cap = cv2.VideoCapture(0)
counter = 0
stage = None

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = pose.process(image)
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        try:
            landmarks = results.pose_landmarks.landmark
            hip = [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x,
                   landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y]
            knee = [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x,
                    landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y]
            ankle = [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x,
                     landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y]
            angle = calculate_angle(hip, knee, ankle)
            cv2.putText(image, str(int(angle)),
                        tuple(np.multiply(knee, [640, 480]).astype(int)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
            if angle > 160:
                stage = "up"
            if angle < 90 and stage == "up":
                stage = "down"
                counter += 1
                play_beep()
                print(f"Squat count: {counter}")
        except:
            pass

        cv2.rectangle(image, (0, 0), (250, 73), (245, 117, 16), -1)
        cv2.putText(image, 'SQUATS', (10, 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1, cv2.LINE_AA)
        cv2.putText(image, str(counter),
                    (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.imshow('Squat Counter', image)
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
            

Squat count: 1
Squat count: 2
Squat count: 3
Squat count: 4
Squat count: 5
Squat count: 6
Squat count: 7
Squat count: 8
Squat count: 9
Squat count: 10
Squat count: 11
Squat count: 12
Squat count: 13
Squat count: 14
Squat count: 15
Squat count: 16
Squat count: 17
Squat count: 18
Squat count: 19


KeyboardInterrupt: 